In [ ]:
import pandas as pd
import numpy as np
import heapq
from sklearn.preprocessing import MinMaxScaler, QuantileTransformer

# load and clean data
df = pd.read_csv('dataset.csv')

df = df[df['duration_ms'] > 0]
df = df[df['tempo'] > 0]
df = df[df['time_signature'] > 0]
df = df.dropna(subset=['artists', 'album_name', 'track_name'])
df = df.drop_duplicates(subset=['track_name', 'artists'])
df = df.reset_index(drop=True)

# feature processing
# energy is right-skewed → fix with QuantileTransformer
qt = QuantileTransformer(output_distribution='uniform', random_state=42)
df['energy_scaled'] = qt.fit_transform(df[['energy']])

# Normalize all features to the same scale
COST_FEATURES = ['valence', 'energy_scaled', 'danceability',
                 'acousticness', 'instrumentalness', 'tempo', 'speechiness']

scaler = MinMaxScaler()
df[COST_FEATURES] = scaler.fit_transform(df[COST_FEATURES])

# Core mood axes (used for heuristic & neighbor search)
CORE = ['valence', 'energy_scaled']

features_core = df[CORE].values
features_cost = df[COST_FEATURES].values

# penalty weights
WEIGHTS = np.array([
    1.0,   # valence          — core mood axis
    1.0,   # energy_scaled    — core mood axis
    0.4,   # danceability     — rhythmic continuity
    0.4,   # acousticness     — prevent style jumps
    0.3,   # instrumentalness — vocal/instrumental consistency
    0.2,   # tempo            — beat continuity
    0.1,   # speechiness      — rap/sung consistency
])

# hueristic function
def h(song_idx, target):
    diff = features_core[song_idx] - target
    return np.sqrt(np.sum(diff ** 2))

# cost function
def cost(idx_a, idx_b, target, remaining_dist, remaining_songs):
    # Weighted feature jump cost
    diff      = features_cost[idx_a] - features_cost[idx_b]
    jump_cost = np.sqrt(np.sum(WEIGHTS * diff ** 2))

    # Ideal step = remaining distance divided evenly
    ideal_step  = remaining_dist / max(remaining_songs, 1)

    # Actual progress made toward target
    actual_step = remaining_dist - h(idx_b, target)

    # Penalize deviation from ideal step size
    step_penalty = abs(actual_step - ideal_step) * 2.0

    return jump_cost + step_penalty

# neighbor search
def get_neighbors(song_idx, target, n=30, ideal_step=0.07):
    dists_to_target = np.linalg.norm(features_core - target, axis=1)
    current_dist    = dists_to_target[song_idx]

    # Where the ideal next song should be in mood space
    ideal_next_dist = max(current_dist - ideal_step, 0)

    # Only consider songs closer to the target than current
    closer_idxs = np.where(dists_to_target < current_dist)[0]

    if len(closer_idxs) == 0:
        # Already at closest point — fall back to nearest neighbors
        dists = np.linalg.norm(features_core - features_core[song_idx], axis=1)
        dists[song_idx] = np.inf
        return np.argsort(dists)[:n]

    # Rank by deviation from ideal next position
    deviation = np.abs(dists_to_target[closer_idxs] - ideal_next_dist)
    top_n     = np.argsort(deviation)[:n]
    return closer_idxs[top_n]

# A* search
def astar_playlist(start_idx, target, max_songs=10, n_neighbors=30, threshold=0.08):
    """
    start_idx   : row index of the starting song in df
    target      : np.array([valence_goal, energy_goal]), range 0~1
    max_songs   : playlist length (user-defined; system ensures smooth transitions)
    n_neighbors : candidate pool size per step
    threshold   : distance threshold for reaching the target mood
    """
    total_dist = h(start_idx, target)
    ideal_step = total_dist / max_songs
    print(f"Total distance: {total_dist:.3f} | Ideal step size: {ideal_step:.3f}")

    start_h = h(start_idx, target)
    queue   = [(start_h, 0.0, start_idx, [start_idx])]
    visited = set()

    best_path = [start_idx]
    best_h    = start_h

    while queue:
        f, g, current, path = heapq.heappop(queue)

        if current in visited:
            continue
        visited.add(current)

        current_h = h(current, target)
        if current_h < best_h:
            best_h    = current_h
            best_path = path

        # condition 1: close enough to target mood
        if current_h < threshold:
            return path

        # condition 2: playlist is long enough
        if len(path) >= max_songs:
            return path

        # remaining songs & distance (used for step_penalty)
        remaining_songs = max_songs - len(path)
        remaining_dist  = current_h

        for neighbor in get_neighbors(current, target, n=n_neighbors,
                                      ideal_step=ideal_step):
            if neighbor not in visited:
                new_g = g + cost(current, neighbor, target,
                                 remaining_dist, remaining_songs)
                new_f = new_g + h(neighbor, target)
                heapq.heappush(queue, (new_f, new_g, neighbor,
                                       path + [neighbor]))

    return best_path

# mood labels
MOOD_MAP = {
    'happy':     np.array([0.85, 0.75]),
    'sad':       np.array([0.15, 0.20]),
    'energetic': np.array([0.60, 0.95]),
    'calm':      np.array([0.55, 0.15]),
    'angry':     np.array([0.10, 0.90]),
    'relaxed':   np.array([0.75, 0.30]),
}

def mood_to_coord(mood_str):
    mood_str = mood_str.lower().strip()
    if mood_str not in MOOD_MAP:
        print(f"Unknown mood '{mood_str}'. Available options: {list(MOOD_MAP.keys())}")
        return None
    return MOOD_MAP[mood_str]

# results
def show_playlist(path, target):
    print(f"\n🎵 Generated Playlist ({len(path)} songs)\n")
    print(f"{'#':<4} {'Track':<35} {'Artist':<25} {'valence':>8} {'energy':>8} {'to target':>10}")
    print("─" * 97)
    for i, idx in enumerate(path):
        row  = df.iloc[idx]
        dist = h(idx, target)
        print(f"{i+1:<4} {str(row['track_name']):<35} "
              f"{str(row['artists']):<25} "
              f"{row['valence']:>8.3f} {row['energy']:>8.3f} {dist:>10.3f}")

# example
if __name__ == '__main__':

    start_idx  = 0
    start_song = df.iloc[start_idx]
    print(f"Starting song: {start_song['track_name']} — {start_song['artists']}")
    print(f"Starting mood: valence={start_song['valence']:.3f}, energy={start_song['energy']:.3f}")

    target_mood = 'energetic'
    target      = mood_to_coord(target_mood)
    print(f"\nTarget mood: {target_mood} → {target}")

    path = astar_playlist(
        start_idx   = start_idx,
        target      = target,
        max_songs   = 10,
        n_neighbors = 30,
        threshold   = 0.08
    )

    show_playlist(path, target)

Starting song: Comedy — Gen Hoshino
Starting mood: valence=0.719, energy=0.461

Target mood: energetic → [0.6  0.95]
Total distance: 0.704 | Ideal step size: 0.070

🎵 Generated Playlist (10 songs)

#    Track                               Artist                     valence   energy  to target
─────────────────────────────────────────────────────────────────────────────────────────────────
1    Comedy                              Gen Hoshino                  0.719    0.461      0.704
2    Sutil dolor                         Benny                        0.538    0.529      0.634
3    Hold On                             Kool & The Gang              0.707    0.601      0.563
4    Let Me In                           Moodymann;Andrés;Sky Covington    0.706    0.660      0.493
5    Long                                Brazilian Girls              0.714    0.714      0.423
6    Be Your Friend‬‬                    Vigiland;Alexander Tidebrink    0.644    0.754      0.352
7    Don't Treat Me Bad 